# Inference Chat RAG Ketenagakerjaan

Notebook ini fokus untuk chat user. Retrieval mengikuti Batch 3: Chroma dense search, BM25, RRF, reranker neural, konteks `citation_text`, dan jawaban dengan sitasi natural. Memori yang dipakai hanya memori jangka pendek beberapa turn terakhir.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag_inference_chat import LegalRAGInference, ShortTermMemory, chat_loop, print_references

print('Project root:', PROJECT_ROOT)


## Inisialisasi Engine

Cell ini memuat embedding model, Chroma collection, BM25 index, reranker, dan LLM. Kalau ingin memakai model lain, set environment variable `RAG_LLM_MODEL_ID` sebelum menjalankan notebook.


In [ ]:
engine = LegalRAGInference(
    use_bm25=True,
    use_reranker=True,
    load_llm=True,
    auto_build_chroma=False,
)
memory = ShortTermMemory(max_turns=4)

print('Device:', engine.device)
print('Chunks:', len(engine.chunks))
print('BM25 aktif:', engine.bm25 is not None)
print('Reranker aktif:', engine.reranker is not None)
print('LLM:', engine.llm_model_id)


## Smoke Test Retrieval

Gunakan cell ini untuk memastikan retrieval sudah jalan sebelum chat panjang.


In [ ]:
docs = engine.retrieve_context('Jika pekerja di-PHK karena pelanggaran berat, berapa pesangonnya?', k=4)
print_references(docs)


## Single Question Mode

Cell ini berguna kalau tidak ingin memakai loop input. Memori tetap dipakai dan akan menyimpan jawaban terakhir.


In [ ]:
query = 'Apakah pekerja kontrak berhak kompensasi saat kontraknya selesai?'
result = engine.answer(query, memory=memory, k=4)
print(result['answer'])
print('\nReferensi:')
print_references(result['references'])


## Chat Interaktif

Ketik pertanyaan user di prompt. Perintah yang tersedia: `clear` untuk menghapus memori, `exit` untuk keluar.


In [ ]:
chat_loop(engine, memory)
